# Classification II: Logistic Regression

**OBJECTIVES**:

- Differentiate between *Regression* and *Classification* problem settings
- Connect Least Squares methods to Classification through Logistic Regression
- Interpret coefficients of the model in terms of probabilities
- Discuss performance of classification model in terms of accuracy
- Understand the effect of an imbalanced target class on model performance

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from scipy import stats

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer, load_digits, load_iris
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline

### Our Motivating Example

Use the form [here](https://forms.gle/qqkwpE3ZXPkEH1NF7) while you work through the notebook.

In [ ]:
default = pd.read_csv('https://raw.githubusercontent.com/jfkoehler/nyu_bootcamp_fa25/refs/heads/main/data/Default.csv', index_col = 0)

In [ ]:
default.info()

In [ ]:
default.head(2)

### Visualizing Default by Continuous Features

In [ ]:
#scatterplot of balance vs. income colored by default status

In [ ]:
sns.scatterplot(data = default, x = 'balance', y = 'income', hue = 'default')

### Considering only `balance` as the predictor



In [ ]:
#create binary default column
default['binary_default'] = np.where(default['default'] == 'No', 0, 1)

In [ ]:
#scatter of Balance vs Default
sns.regplot(data = default, x = 'balance', y = 'binary_default')

### The Sigmoid aka Logistic Function


$$y = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x)}}$$

In [ ]:
#domain
x = np.arange(0, 2800, .1)

In [ ]:
sns.scatterplot(data = default, x = 'balance', y = 'binary_default', hue = 'default')
plt.plot(x, 1/(1 + np.exp(-0.0055*x +10.76105259)), '--', color = 'black', label = 'sigmoid')
plt.legend();

In [ ]:
def make_plot(m,b):
    sns.scatterplot(data = default, x = 'balance', y = 'binary_default', hue = 'default')
    plt.plot(x, 1/(1 + np.exp(-m*x +b)), '--', color = 'black', label = 'sigmoid')
    plt.legend();

In [ ]:
interact(make_plot, m = widgets.FloatSlider(description = r'$\beta_1$', min = 0, max = 0.1, step = 0.001),
         b = widgets.FloatSlider(description = r'$\beta_0$', min = 9, max = 11, step = 0.1));

### Example: Hypothetical Models

Below, the array `y` represents a target of true values and `yhat1` and `yhat2` represent two different model predictions.  Which model's predictions are better?

In [ ]:
y = np.array([1, 1, 1, 0, 0])
yhat1 = np.array([1, 0, 0, 0, 1])
yhat2 = np.array([1, 1, 1, 1, 0])

In [ ]:
pd.DataFrame({'y': y, 'yhat1': yhat1, 'yhat2': yhat2})

### Quantifying Loss

In regression, we understood the `LinearRegression` model as one seeking to minimize the mean squared error of a linear model with different slope and intercept.  With classification, we can also understand the quality of a model given an objective or **loss** function.  One such classification loss functions is **log loss** and through minimizing this loss with our sigmoid model we get the **Logistic Regression** model.

$$
\text{Log Loss or Cross Entropy}: \ell_{k}=-y_{k}\ln p_{k}-(1-y_{k})\ln(1-p_{k})
$$

In [ ]:
from sklearn.metrics import log_loss

In [ ]:
log_loss(y, yhat1)

In [ ]:
log_loss(y, yhat2)

### Usage should seem familiar

Fit a `LogisticRegression` estimator from `sklearn` on the features:

```python 
X = default[['balance']]
y = default['binary_default']
```

In [ ]:
#instantiate
clf = LogisticRegression()

In [ ]:
#define X and y
X = default[['balance']]
y = default['default']

In [ ]:
#train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 22)

In [ ]:
#fit on the train


In [ ]:
#examine train and test scores


### Evaluating the Classifier

In `scikitlearn` the primary default evalution metric is **accuracy** or percent correct.  We still need to compare this to our baseline -- typically predicting the most frequently occurring class.  Further, you can investigate the mistakes made with each class by looking at the **Confusion Matrix**.  A quick visualization of this is had using the `ConfusionMatrixDisplay`.

In [ ]:
#baseline -- most frequently occurring class
y_train.value_counts(normalize = True)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

In [ ]:
#from estimator
ConfusionMatrixDisplay.from_estimator(clf, X_test, y_test)

### Interpreting Output of the Model

The version of the logistic we have just developed is actually:

$$ y = \frac{e^{ax + b}}{1 + e^{ax + b}} $$

Its output represents probabilities of being labeled the positive class in our example.  This means that we can interpret the output of the above function using our parameters, remembering that we used the `balance` feature to predict `default`.

In [ ]:
def predictor(x):
    line = clf.coef_[0]*x + clf.intercept_
    return np.e**line/(1 + np.e**line)

In [ ]:
#predict 1000
predictor(1000)

In [ ]:
#predict 2000
predictor(2000)

In [ ]:
#estimator has this too
clf.predict_proba(np.array([[1000]]))

In [ ]:
clf.predict(np.array([[1000], [2000]]))

**PROBLEM**: Given the probability of default below, can you understand what the code is doing?  Is the model performance different?

In [ ]:
probability_default = clf.predict_proba(X)[:, 1]

In [ ]:
new_predictions = np.where(probability_default > .3, 'Yes', 'No')

In [ ]:
ConfusionMatrixDisplay.from_predictions(y, new_predictions)

In [ ]:
features = ['balance', 'income', 'student_binary']
X = default.loc[:, features]
y = default['binary_default']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=22)

In [ ]:
clf = LogisticRegression().fit(X_train, y_train)

In [ ]:
clf.score(X_train, y_train)

In [ ]:
clf.score(X_test, y_test)

**Predictions**:

- student: yes
- balance: 1,500 dollars
- income: 40,000 dollars

In [ ]:
ex1 = np.array([[1500, 40_000, 1]])
#predict probability
clf.predict_proba(ex1)

- student: no
- balance: 1,500 dollars
- income: 40,000 dollars

In [ ]:
ex2 = np.array([[1500, 40_000, 0]])
#predict probability
clf.predict_proba(ex2)

### This is similar to our multicollinearity in regression; we will call it confounding

<center>
<img src = 'https://github.com/jfkoehler/nyu_bootcamp_fa24/blob/main/images/default_confound.png?raw=true' />
</center>

#### Using `scikitlearn` and its `Pipeline`

From the original data, to build a model involved:

1. One hot or dummy encoding the categorical feature.
2. Standard Scaling the continuous features
3. Building Logistic model

we can accomplish this all with the `Pipeline`, where the first step is a `make_column_transformer` and the second is a `LogisticRegression`.  

In [ ]:
from sklearn.pipeline import Pipeline 
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(default[['student', 'income', 'balance']], default['default'],
                                                   random_state = 22)

In [ ]:
# create OneHotEncoder instance
ohe = OneHotEncoder(drop = 'first')

In [ ]:
# create StandardScaler instance
sscaler = StandardScaler()

In [ ]:
# make column transformer
transformer = make_column_transformer((ohe, ['student']), 
                                     remainder = sscaler,
                                     force_int_remainder_cols=False)

In [ ]:
# logistic regressor
clf = LogisticRegression()

In [ ]:
# pipeline
pipe = Pipeline([('transform', transformer), 
                 ('model', clf)])

In [ ]:
# fit it
pipe.fit(X_train, y_train)

In [ ]:
# score on train and test
print(f'Train Score: {pipe.score(X_train, y_train)}')
print(f'Test Score: {pipe.score(X_test, y_test)}')

In [ ]:
pipe.named_steps['model'].coef_

#### Problem

Below, a dataset on bank customer churn is loaded and displayed.  Your objective is to predict `Exited` or not.  Use `CreditScore`, `Gender`, `Age`, `Tenure`, and `Balance` as predictors.  Examine the confusion matrix display.  Was your classifier better at predicting exits or non-exits?

In [ ]:
from sklearn.datasets import fetch_openml

In [ ]:
bank_churn = fetch_openml(data_id = 43390).frame

In [ ]:
bank_churn.head()

In [ ]:
#create train/test split -- random_state = 11
X = bank_churn.drop(columns = 'Exited')
y = bank_churn['Exited']
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=11)

#### Practice

In [ ]:
from sklearn.datasets import load_breast_cancer

In [ ]:
cancer = load_breast_cancer(as_frame=True).frame

In [ ]:
cancer.head(3)

In [ ]:
# use all features


In [ ]:
# train/test split -- random_state = 42


In [ ]:
# pipeline to scale then knn


In [ ]:
# pipeline to scale then logistic


In [ ]:
# fit knn


In [ ]:
# fit logreg


In [ ]:
# compare confusion matrices on test data
